In [10]:
using JuMP
using Gurobi
using Random
using Dualization
using Plots
using DataFrames
using CSV
import XLSX
import JSON

current_directory = @__DIR__
functions_directory = joinpath(current_directory, "functions")
data_dir = joinpath(current_directory, "data")
results_dir = joinpath(current_directory, "results")

# Include all the function files
# include(joinpath(functions_directory, "create_check_params.jl"))
# include(joinpath(functions_directory, "deterministic_equivalent.jl"))
# include(joinpath(functions_directory, "generate_cuts_from_dual.jl"))
include(joinpath(functions_directory, "load_model_starting_points.jl"))
include(joinpath(functions_directory, "load_model_starting_points_OLD.jl"))
include(joinpath(functions_directory, "initialize_parameters.jl"))
include(joinpath(functions_directory, "process_scenario_data.jl"))
include(joinpath(functions_directory, "process_scenario_data_n_selected.jl"))
# include(joinpath(functions_directory, "save_L_shaped_results.jl"))
include(joinpath(functions_directory, "select_random_scenarios.jl"))
include(joinpath(functions_directory, "create_vaccine_data.jl"))
# include(joinpath(functions_directory, "sub_problem.jl"))
# include(joinpath(functions_directory, "master_problem.jl"))
include(joinpath(functions_directory, "create_vaccine_data_MMR_only.jl"))

create_vaccine_data_MMR_only (generic function with 1 method)

In [11]:
A, V, A_v, P, P_v, V_a, V_p, P_a, A_p, capacity_category, vaccine_category, antigen_category = create_vaccine_data()

(["Measles", "Mumps", "Rubella", "Diphtheria", "Tetanus", "Pertussis", "Hepatitis_B", "Hib", "Polio", "HPV", "Rotavirus", "PCV"], ["M", "MR", "MMR", "TT", "HepB", "Hib", "IPV", "OPV", "DT", "Td", "DTwP", "DTwP-Hib", "Penta", "Hexa", "HPV", "Rotavirus", "PCV"], Dict("MMR" => ["Measles", "Mumps", "Rubella"], "Td" => ["Diphtheria", "Tetanus"], "PCV" => ["PCV"], "Rotavirus" => ["Rotavirus"], "DTwP-Hib" => ["Diphtheria", "Tetanus", "Pertussis", "Hib"], "IPV" => ["Polio"], "Hexa" => ["Diphtheria", "Tetanus", "Pertussis", "Hepatitis_B", "Hib", "Polio"], "TT" => ["Tetanus"], "HPV" => ["HPV"], "MR" => ["Measles", "Rubella"]…), ["AJ_Vaccines", "BB_NCIPD", "China_National", "Bharat_Biotech", "Bilthoven", "Biological_E", "GSK", "Haffkine_Bio", "LG_Chem", "Merck_Sharp", "Panacea_Biotec", "PT_Bio", "Sanofi", "Serum_Institute", "Pfizer"], Dict("MMR" => ["Serum_Institute", "GSK"], "Td" => ["Serum_Institute", "PT_Bio", "BB_NCIPD", "Biological_E"], "PCV" => ["Serum_Institute", "GSK", "Pfizer"], "Rotavir

In [12]:
starting_points_vect_F, starting_points_vect_I, starting_points_vect_S = load_model_starting_points(data_dir, 1, 1000, A, V)

(Any[("Measles", 1, 3), ("Mumps", 1, 3), ("Rubella", 1, 3), ("Diphtheria", 1, 1), ("Tetanus", 1, 1), ("Pertussis", 1, 1), ("Hib", 1, 1), ("Hepatitis_B", 1, 1), ("Polio", 1, 1), ("HPV", 1, 5), ("Rotavirus", 1, 1), ("PCV", 1, 3)], Any[("Penta", 398056.0), ("OPV", 27969.0), ("IPV", 410128.0), ("PCV", 590380.0), ("M", 280764.0), ("MR", 879375.0), ("MMR", 83172.0), ("TT", 7683.0), ("HepB", 3279.0), ("Hib", 2829.0), ("DT", 116834.0), ("Td", 5299.0), ("DTwP", 30290.0), ("DTwP-Hib", 43272.0), ("Hexa", 86544.0), ("HPV", 242286.0), ("Rotavirus", 156727.0)], Any[("Diphtheria", 0.0), ("Hib", 0.0), ("Measles", 0.0), ("Mumps", 0.0), ("PCV", 0.0), ("Pertussis", 0.0), ("Polio", 0.0), ("Rotavirus", 0.0), ("Rubella", 0.0), ("Tetanus", 0.0), ("Hepatitis_B", 0.0), ("HPV", 0.0)])

In [13]:
starting_points_vect_F2, starting_points_vect_I2, starting_points_vect_S2 = load_model_starting_points_OLD(data_dir, 1, 1000, A, V)

(Any[("Measles", 1, 3), ("Mumps", 1, 3), ("Rubella", 1, 3), ("Diphtheria", 1, 1), ("Tetanus", 1, 1), ("Pertussis", 1, 1), ("Hib", 1, 1), ("Hepatitis_B", 1, 1), ("Polio", 1, 1), ("HPV", 1, 5), ("Rotavirus", 1, 1), ("PCV", 1, 3)], Any[("Penta", 398056.0), ("OPV", 27969.0), ("IPV", 410128.0), ("PCV", 590380.0), ("M", 280764.0), ("MR", 879375.0), ("MMR", 83172.0), ("TT", 7683.0), ("HepB", 3279.0), ("Hib", 2829.0), ("DT", 116834.0), ("Td", 5299.0), ("DTwP", 30290.0), ("DTwP-Hib", 43272.0), ("Hexa", 86544.0), ("HPV", 242286.0), ("Rotavirus", 156727.0)], Any[("Diphtheria", 0.0), ("Hib", 0.0), ("Measles", 0.0), ("Mumps", 0.0), ("PCV", 0.0), ("Pertussis", 0.0), ("Polio", 0.0), ("Rotavirus", 0.0), ("Rubella", 0.0), ("Tetanus", 0.0), ("Hepatitis_B", 0.0), ("HPV", 0.0)])

In [16]:
def split_bounds(data, exclude_first_n=0):
    """Splits the given data into upper and lower bound lists, 
    excludes the first N points, and plots them."""
    lb, ub, runtime = data['lb'][exclude_first_n:], data['ub'][exclude_first_n:], data['run_time']
   
    return lb, ub, runtime  # Return the split data if needed

data = {"lb":[4.94895182975306e6,3.500596614835796e7,3.5789299423326045e7,3.6111377005832165e7,3.694878008957938e7,3.744444257676009e7,3.7868438634155415e7,3.808466206450485e7,3.8253303190938085e7,4.97834196905268e7,5.164999034735819e7,5.338224858740705e7,5.436221639679021e7,5.529904687928803e7,5.620319626876248e7,5.668243734798529e7,5.706452492004621e7,5.7445251203016005e7,5.744525120301596e7,5.764864645149138e7,5.805533901014486e7,5.8079469082257554e7,5.81374885884487e7,5.824089331276479e7,5.85816988260119e7,5.878377163477369e7,5.8934401357123524e7,5.9289432705873236e7,5.9392980972217955e7,5.9501100569025904e7,5.963278784439027e7,5.979913912109632e7,6.002714739182063e7,6.010206684341927e7,6.017110102786439e7,6.0286547062796086e7,6.0420981779454455e7,6.049344911338119e7,6.053125490445737e7,6.058981387923269e7,6.063403038209289e7,6.066503675451084e7,6.075430055319485e7,6.0776929087432206e7,6.081245506099155e7,6.084978843555371e7,6.090240071918347e7,6.093743489978373e7,6.097148065244841e7,6.100288339638662e7,6.103791716606158e7,6.107889328525558e7],"ub":[4.0475615034901276e7,3.946846471222307e7,3.946846471222307e7,3.893334098056817e7,3.893334098056817e7,3.893334098056817e7,3.893334098056817e7,3.893334098056817e7,3.893334098056817e7,2.0654315189056385e8,1.4416439790957856e8,9.594617972637361e7,9.07174533232577e7,8.553251054187824e7,7.35032676919936e7,6.947571597880194e7,6.947571597880194e7,6.947571597880194e7,6.56107996246348e7,6.56107996246348e7,6.56107996246348e7,6.460546778232041e7,6.407999573550123e7,6.352117134165267e7,6.352117134165267e7,6.352117134165267e7,6.352117134165267e7,6.352117134165267e7,6.352117134165267e7,6.352117134165267e7,6.352117134165267e7,6.352117134165267e7,6.352117134165267e7,6.300968629250564e7,6.2314020100855984e7,6.2314020100855984e7,6.2314020100855984e7,6.2314020100855984e7,6.228519152844114e7,6.228519152844114e7,6.228519152844114e7,6.228519152844114e7,6.228519152844114e7,6.216223747447846e7,6.216223747447846e7,6.216223747447846e7,6.216223747447846e7,6.216223747447846e7,6.2114494255144954e7,6.1892270945103906e7,6.1892270945103906e7,6.164451765736767e7],"run_time":1561.6129970550537}
data2 = {"lb":[3.492490538275682e6,2.4340471844800804e7,2.4839669493489724e7,2.6740446968979396e7,2.6842599086542673e7,2.7178903559546504e7,2.730854884335058e7,2.74559036021694e7,2.7455903602169402e7,3.4180134738721296e7,3.635149353146188e7,3.816666742338318e7,3.942526139896412e7,4.115197572930397e7,4.243653651142752e7,4.3243539317365766e7,4.3949125028908536e7,4.5368982297674105e7,4.653492062685437e7,4.7104530359119676e7,4.815666639275107e7,4.882492254800674e7,5.000306867918983e7,5.000306867918971e7,5.000306867918986e7,5.000306867918978e7,5.0003068679189906e7,5.000306867918991e7,5.000306867918987e7,5.429805340609202e7,5.972380476530717e7,5.972380476530722e7,5.9723804765307166e7,5.972380476530713e7,5.984188187960105e7,6.424674267354986e7,6.424674267354978e7,6.4246742673549816e7,6.424674267354975e7,6.4246742673549764e7,6.795980099188925e7,7.30769683274009e7,7.835692929476216e7],"ub":[3.2497145183648303e7,2.9988892475417625e7,2.9082306150708888e7,2.8229435314985253e7,2.8229435314985253e7,2.8229435314985253e7,2.8229435314985253e7,2.8166556202787153e7,2.7872114769658726e7,4.1387073335299057e8,2.0259498034798774e8,1.542660279447951e8,1.542660279447951e8,9.948886793536769e7,9.948886793536769e7,9.948886793536769e7,8.31215864281741e7,8.31215864281741e7,8.31215864281741e7,8.31215864281741e7,8.31215864281741e7,7.81039504574755e7,7.81039504574755e7,7.81039504574755e7,7.81039504574755e7,7.81039504574755e7,7.81039504574755e7,7.81039504574755e7,7.81039504574755e7,7.81039504574755e7,7.81039504574755e7,7.81039504574755e7,7.81039504574755e7,7.81039504574755e7,7.81039504574755e7,7.81039504574755e7,7.81039504574755e7,7.81039504574755e7,7.81039504574755e7,7.81039504574755e7,7.81039504574755e7,7.81039504574755e7,7.81039504574755e7],"run_time":11507.675091028214}
lb, ub, runtime = split_bounds(data)
lb2, ub2, runtime2 = split_bounds(data2)

In [ ]:
import matplotlib.pyplot as plt

# Create figure and plot data
plt.figure(figsize=(10, 5))
# plt.plot(range(len(lb)), lb, label="Lower Bound", linestyle='dashed', marker='o')
# plt.plot(range(len(ub)), ub, label="Upper Bound", linestyle='solid', marker='s')
# Plot first set
plt.plot(range(len(lb)), lb, label="Lower Bound - 1 MP Scenario", linestyle='dashed', marker='o')
plt.plot(range(len(ub)), ub, label="Upper Bound - 1 MP Scenario", linestyle='solid', marker='o')

# Plot second set
plt.plot(range(len(lb2)), lb2, label="Lower Bound - 3 MP Scenario", linestyle='dotted', marker='x')
plt.plot(range(len(ub2)), ub2, label="Upper Bound - 3 MP Scenario", linestyle='dashdot', marker='x')

plt.xlabel("Index")
plt.ylabel("Value")
plt.title("UNICEF-GAVI Phase 2 convergence - 1 v 3 MP Scenario")
plt.legend()
plt.grid(True)
plt.savefig("unicef_gavi_Convergence_phase_two.pdf")
plt.show()
